## 5-Day Temperature Forcast
This notebook uses the preocessed weather data to train a multi optput model that predicts the next 5 days of temerature.

In [346]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pickle
import plotly.express as px

import plotly.io as pio
pio.renderers.default = "notebook_connected+vscode"

In [347]:
df = pd.read_csv("data/features.csv")
df.columns

Index(['cloud_cover', 'sunshine', 'global_radiation', 'max_temp', 'mean_temp',
       'min_temp', 'precipitation', 'pressure', 'snow_depth', 'year',
       'temp_tomorrow', 'season_spring', 'season_summer', 'season_winter',
       'temp_yesterday', 'rain_yesterday', 'sunshine_yesterday',
       'pressure_yesterday', 'cloud_yesterday', 'temp_last3', 'pressure_last3',
       'sun_last3', 'cloud_last3', 'rain_last3', 'temp_last5',
       'pressure_last5', 'sun_last5', 'cloud_last5', 'rain_last5'],
      dtype='object')

In [348]:
df = df.drop(columns=["temp_tomorrow"])

In [349]:
# The 5 target column, one for each future day
df["temp_day1"] = df["mean_temp"].shift(-1)
df["temp_day2"] = df["mean_temp"].shift(-2)
df["temp_day3"] = df["mean_temp"].shift(-3)
df["temp_day4"] = df["mean_temp"].shift(-4)
df["temp_day5"] = df["mean_temp"].shift(-5)

df = df.dropna()

#### Decision Tree For Feature Importance
Again to determine if hthe importance has shifted from the previous model for day 5.

In [350]:
X = df.drop(columns=["temp_day5", "temp_day4", "temp_day3", "temp_day2", "temp_day1"])
 
# Targets
y = df["temp_day5"]

# Splitting
train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [351]:
tree = DecisionTreeRegressor(random_state=42)
tree.fit(X_train, y_train)

y_test_pred = tree.predict(X_test)

In [352]:
features = X.columns
importance = pd.Series(tree.feature_importances_, index=X.columns).sort_values(ascending=False)
importance = importance.reset_index(name="importance")

In [353]:
importance.T

,0,1,2,3,4,5,6,7,8,9,...,18,19,20,21,22,23,24,25,26,27
index,max_temp,season_summer,season_winter,temp_last5,year,global_radiation,sun_last5,pressure,pressure_last3,pressure_last5,...,sunshine_yesterday,rain_last3,cloud_last5,cloud_last3,precipitation,rain_yesterday,cloud_yesterday,cloud_cover,season_spring,snow_depth
importance,0.615619,0.073311,0.050164,0.040075,0.020722,0.014699,0.014009,0.013074,0.012449,0.012446,...,0.009114,0.008475,0.007818,0.005448,0.005076,0.00474,0.004583,0.003296,0.000779,0.000082


In [354]:
fig_importance = px.bar(importance, x="index", y="importance", title="Feature Importance", log_y=True)
fig_importance.show()

In [355]:
importance

,index,importance
0,max_temp,0.615619
1,season_summer,0.073311
2,season_winter,0.050164
3,temp_last5,0.040075
4,year,0.020722
5,global_radiation,0.014699
6,sun_last5,0.014009
7,pressure,0.013074
8,pressure_last3,0.012449
9,pressure_last5,0.012446


In [356]:
final_features = ["max_temp", "season_summer", "season_winter", "temp_last5", "year"]


X = df[final_features]
y = df[["temp_day1", "temp_day2", "temp_day3", "temp_day4", "temp_day5"]]

X_train, X_test, y_train,  y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [357]:
model = MultiOutputRegressor(LinearRegression())
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [358]:
results = []
for i, col in enumerate(['Day 1', 'Day 2', 'Day 3', 'Day 4', 'Day 5']):
    mae  = mean_absolute_error(y_test.iloc[:, i], y_pred[:, i])
    r2   = r2_score(y_test.iloc[:, i], y_pred[:, i])
    results.append({'Day': col, 'MAE': round(mae, 3), 'R²': round(r2, 3)})
results_df = pd.DataFrame(results)
print(results_df)

fig = px.line(results_df, x='Day', y=['MAE', 'R²'], markers=True,
              title='Multi-Output Model: Accuracy by Forecast Day')
fig.show()

     Day    MAE     R²
0  Day 1  1.073  0.944
1  Day 2  1.495  0.883
2  Day 3  1.912  0.818
3  Day 4  2.169  0.769
4  Day 5  2.311  0.735


* MAE going up: The further into the furture the more off the prediction get. Each extra day adds roughly +0.3C of error.
* R2 going up: The model is less able to explain of the variation in the data as we predict further into the future. Worth nothing they are gentle steady sloped, not a cliff, which is good.

Overall the shape of this chart is exactly what a healthy multi output model looks like.

In [359]:
final_features = ["max_temp", "season_summer", "season_winter", "temp_last5", "year"]

with open("models/multi_temperature_model.pkl", "wb") as f:
    pickle.dump(model, f)
    
with open("models/multi_temperature_features.pkl", "wb") as f:
    pickle.dump(final_features, f)

***

In [360]:
with open("models/multi_temperature_model.pkl", "rb") as f:
    loaded_model = pickle.load(f)

In [361]:
loaded_model

,estimator,LinearRegression()
,n_jobs,None
,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [362]:
user_data = {
 "max_temp": 10.0,        # Max temp yesterday in °C
 "year": 2026,           # Current year  
 "last1": 9.375,           # 1 day before yesterday mean temp in °C 
 "last2": 6.125,           # 2 days before mean temp
 "last3": 6.75,           # 3 days before mean temp 
 "last4": 10.75,           # 4 days before mean temp
 "last5": 9.625,           # 5 days before mean temp
 "season_summer": False,
 "season_winter": False   # If it is True, than it's winter, otherwise it is summer.
}

In [363]:
# Mean temperature (°C)

# 5-day rolling mean temperature (°C)
temp_last5 = np.mean([
    user_data["last1"],
    user_data["last2"],
    user_data["last3"],
    user_data["last4"],
    user_data["last5"]
])
# Pressure: mbar → Pa
# Cloud: % -> Okat
input_row = {
    "max_temp": user_data["max_temp"],
    "season_summer": user_data["season_summer"],
    "season_winter": user_data["season_winter"],
    "temp_last5": temp_last5,
    "year": user_data["year"]
}

In [364]:
X_input = pd.DataFrame([input_row])

In [365]:
X_input

,max_temp,season_summer,season_winter,temp_last5,year
0,10.0,False,False,8.525,2026


In [366]:
prediction = loaded_model.predict(X_input)[0]
print(f"Predicted temperature tomorrow: {prediction} °C")

Predicted temperature tomorrow: [7.17783705 7.62897437 8.24535666 8.7580604  9.07657723] °C
